# Capítulo 19 - Arquitetura Multi-Agente On-Premise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap19_multi_agente.ipynb)

Notebook **prático e autoral**: reproduz, executável e em CPU, os padrões multi-agente do **AI-Orchestrator** — classificação de intenção em camadas, fan-out/fan-in, Tool Registry com escopo por domínio e Circuit Breaker. Onde a stack real (Ollama/Qdrant/serviços HTTP) seria necessária, usamos *mocks* locais; cada célula aponta o arquivo real.

---

## Atribuição

Material **autoral** do projeto AI-Orchestrator (`gateway/`), por Allan Eric Jepsen, para *LLM On-Premise: Guia Prático*. Espelha: `gateway/graph.py`, `gateway/router.py`, `gateway/agents.py`, `gateway/tools/registry.py`, `gateway/tools/circuit.py`.

**Referências:** capítulo `livro/cap19_multi_agente.md`; LangGraph; framework ADD (Model vs Harness).

---

## Parte 1 — As 4 topologias e o framework ADD

O AI-Orchestrator usa **4 agentes de domínio** (financas, rh, estoque, vendas), cada um com ferramentas isoladas. O princípio **ADD** (*Agent Design Decisions*) diz: prefira **Harness** (lógica determinística) a **Model** (LLM) sempre que possível — mais barato, previsível e seguro.

O fluxo do grafo: `sanitize → enrich → classify → confirm → dispatch → synthesize`.

## Parte 2 — Classificador de intenção em 3 camadas

1. **Semântico** (Qdrant + embeddings): consenso de vizinhos do golden.
2. **LLM** (prompt): decide quando o semântico não tem consenso.
3. **Léxico** (determinístico): fallback por keyword se tudo falhar.

Harness-first: o léxico nunca depende de rede. Abaixo, uma versão reduzida (camadas semântica e LLM são *stubs* para rodar offline).

In [1]:
import re, unicodedata

def _normalize(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))

_KW = {
    "financas": {"fatura", "boleto", "fluxo de caixa"},
    "rh": {"salario", "ferias", "funcionario"},
    "estoque": {"estoque", "sku", "inventario"},
    "vendas": {"comissao", "pedido", "desconto", "vendedor"},
}

def lexical_classify(q):
    n = _normalize(q)
    hits = [d for d, kws in _KW.items()
            if any(re.search(rf"(?<![a-z]){re.escape(k)}", n) for k in kws)]
    return hits

def semantic_stub(q):       # no projeto: Qdrant + consenso de vizinhos
    return None             # devolve None => sem consenso => proxima camada

def llm_stub(q):            # no projeto: classificacao por prompt no Ollama
    return None

def classify(q):
    for layer, fn in [("semantico", semantic_stub), ("llm", llm_stub), ("lexico", lexical_classify)]:
        res = fn(q)
        if res:
            return layer, res
    return "nenhum", []

for q in ["qual a comissao do pedido?", "quando sai meu salario?", "ola tudo bem?"]:
    print(f"{q!r:42} -> {classify(q)}")

'qual a comissao do pedido?'               -> ('lexico', ['vendas'])
'quando sai meu salario?'                  -> ('lexico', ['rh'])
'ola tudo bem?'                            -> ('nenhum', [])


## Parte 3 — Fan-out / Fan-in

Quando a query toca **vários domínios**, o orquestrador faz *fan-out*: despacha os agentes **em paralelo** (`ThreadPoolExecutor`) e depois *fan-in*: sintetiza as respostas numa só. Veja `gateway/graph.py` (dispatch/synthesize).

In [2]:
from concurrent.futures import ThreadPoolExecutor
import time

def run_agent(domain, query):           # no projeto: chama o microsservico HTTP do dominio
    time.sleep(0.05)                     # simula latencia
    return domain, f"[{domain}] resposta para: {query}"

def fan_out(domains, query):
    with ThreadPoolExecutor(max_workers=4) as ex:
        return list(ex.map(lambda d: run_agent(d, query), domains))

def fan_in(parts):                       # sintese (no projeto: 1 chamada LLM)
    return "Sintese:\n" + "\n".join(f"  - {body}" for _d, body in parts)

t0 = time.time()
parts = fan_out(["vendas", "estoque"], "tem desconto nesse SKU?")
print(fan_in(parts))
print(f"\nlatencia paralela: {time.time()-t0:.3f}s (2 agentes ~ 0.05s, nao 0.10s)")

Sintese:
  - [vendas] resposta para: tem desconto nesse SKU?
  - [estoque] resposta para: tem desconto nesse SKU?

latencia paralela: 0.051s (2 agentes ~ 0.05s, nao 0.10s)


## Parte 4 — Tool Registry com escopo por domínio (least-privilege)

Cada agente só enxerga as ferramentas do **seu** domínio — um agente de RH nunca recebe ferramentas de finanças. Isso é *least-privilege tool scope* (ver `gateway/tools/registry.py`). No projeto, as tools são descobertas via **OpenAPI** dos microsserviços; aqui registramos manualmente.

In [3]:
class ToolRegistry:
    def __init__(self):
        self._by_domain = {}
    def register(self, domain, name):
        self._by_domain.setdefault(domain, []).append(name)
    def tools_for(self, domain):
        return list(self._by_domain.get(domain, []))

reg = ToolRegistry()
reg.register("rh", "consultar_funcionario")
reg.register("rh", "solicitar_ferias")
reg.register("financas", "consultar_saldo")
reg.register("financas", "aprovar_pagamento")

print("Agente RH ve:      ", reg.tools_for("rh"))
print("Agente Financas ve:", reg.tools_for("financas"))
assert "aprovar_pagamento" not in reg.tools_for("rh"), "RH nunca acessa tools de financas"
print("\nOK: isolamento por dominio garantido.")

Agente RH ve:       ['consultar_funcionario', 'solicitar_ferias']
Agente Financas ve: ['consultar_saldo', 'aprovar_pagamento']

OK: isolamento por dominio garantido.


## Parte 5 — Resiliência: Circuit Breaker por domínio

Se um microsserviço começa a falhar (timeout/5xx), o **Circuit Breaker** *abre* após N falhas consecutivas e passa a devolver resposta degradada imediata (sem martelar o serviço caído). Após um *cooldown*, entra em *half-open*: 1 *probe* decide se fecha ou reabre. Código espelhado de `gateway/tools/circuit.py` (com relógio injetável para o demo).

In [4]:
# Espelho de gateway/tools/circuit.py (reduzido, relogio injetavel)
CLOSED, OPEN, HALF_OPEN = "closed", "open", "half_open"

class CircuitBreaker:
    def __init__(self, threshold=3, cooldown_s=30.0, clock=None):
        self._th, self._cd = threshold, cooldown_s
        self._clock = clock or (lambda: 0.0)
        self._d = {}
    def _e(self, dom):
        return self._d.setdefault(dom, {"state": CLOSED, "failures": 0, "opened_at": 0.0})
    def state(self, dom):
        return self._e(dom)["state"]
    def allow(self, dom):
        e = self._e(dom)
        if e["state"] == CLOSED:
            return True
        if e["state"] == OPEN and self._clock() - e["opened_at"] >= self._cd:
            e["state"] = HALF_OPEN
            return True
        return e["state"] == HALF_OPEN and False
    def record_success(self, dom):
        self._e(dom).update(state=CLOSED, failures=0)
    def record_failure(self, dom):
        e = self._e(dom)
        if e["state"] == HALF_OPEN:
            e.update(state=OPEN, opened_at=self._clock()); return
        e["failures"] += 1
        if e["state"] == CLOSED and e["failures"] >= self._th:
            e.update(state=OPEN, opened_at=self._clock())

now = {"t": 0.0}
cb = CircuitBreaker(threshold=3, cooldown_s=30.0, clock=lambda: now["t"])
for i in range(3):
    cb.record_failure("estoque")
print("apos 3 falhas:", cb.state("estoque"), "| allow:", cb.allow("estoque"))
now["t"] = 31.0
print("apos cooldown:  allow(probe):", cb.allow("estoque"), "->", cb.state("estoque"))
cb.record_success("estoque")
print("probe ok:      ", cb.state("estoque"))

apos 3 falhas: open | allow: False
apos cooldown:  allow(probe): True -> half_open
probe ok:       closed


## Conclusão e mapa para o código

| Conceito | Arquivo real |
|---|---|
| Grafo (sanitize→...→synthesize) | `gateway/graph.py` |
| Classificação 3 camadas | `gateway/router.py`, `gateway/semantic_router.py` |
| Fan-out/Fan-in | `gateway/graph.py`, `gateway/agents.py` |
| Tool Registry (OpenAPI, escopo) | `gateway/tools/registry.py` |
| Circuit Breaker | `gateway/tools/circuit.py` |

**Princípio transversal:** Harness antes de Model, isolamento por domínio, degradação graceful. Texto completo em `livro/cap19_multi_agente.md`.